In [2]:

import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [3]:
df = pd.read_csv('../data/adult.csv')
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [4]:
df.replace([' ?', '?'], np.nan, inplace=True)
df.dropna(inplace=True)

df['income'] = df['income'].map({
    '<=50K': 0, '>50K': 1
})

Wygląda to tak, że eduaction num i edukacją nie są obie na raz potrzebne

fnlwgt tak jakies losowe liczby chyba

In [5]:
print(df.groupby('education')['education.num'].nunique())


education
10th            1
11th            1
12th            1
1st-4th         1
5th-6th         1
7th-8th         1
9th             1
Assoc-acdm      1
Assoc-voc       1
Bachelors       1
Doctorate       1
HS-grad         1
Masters         1
Preschool       1
Prof-school     1
Some-college    1
Name: education.num, dtype: int64


Faktycznie tak jest

In [6]:
# Usunięcie zbędnych kolumn (OKAZAŁO SIĘ, że tych cech i tak jest za dużo, więc więcej usunałem)
if 'education' in df.columns:
    df = df.drop('education', axis=1)
# if 'education.num' in df.columns:
    # df = df.drop('education.num', axis=1)
if 'fnlwgt' in df.columns:
    df = df.drop('fnlwgt', axis=1)
# if 'race' in df.columns:
#     df = df.drop('race', axis=1)
# if 'relationship' in df.columns:
#     df = df.drop('relationship', axis=1)
# if 'marital.status' in df.columns:
#     df = df.drop('marital.status', axis=1)
# if 'native.country' in df.columns:
#     df = df.drop('native.country', axis=1)

In [7]:
X = df.drop('income', axis=1)
y = df['income'].astype(int)

In [8]:
# Przekształcenie zmiennych kategorycznych
categorical_cols = X.select_dtypes(include=['object']).columns
X_processed = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print(f"Liczba kolumn po get_dummies: {X_processed.shape[1]}")
print(f"Kolumny to: {X_processed.columns}")

Liczba kolumn po get_dummies: 80
Kolumny to: Index(['age', 'education.num', 'capital.gain', 'capital.loss',
       'hours.per.week', 'workclass_Local-gov', 'workclass_Private',
       'workclass_Self-emp-inc', 'workclass_Self-emp-not-inc',
       'workclass_State-gov', 'workclass_Without-pay',
       'marital.status_Married-AF-spouse', 'marital.status_Married-civ-spouse',
       'marital.status_Married-spouse-absent', 'marital.status_Never-married',
       'marital.status_Separated', 'marital.status_Widowed',
       'occupation_Armed-Forces', 'occupation_Craft-repair',
       'occupation_Exec-managerial', 'occupation_Farming-fishing',
       'occupation_Handlers-cleaners', 'occupation_Machine-op-inspct',
       'occupation_Other-service', 'occupation_Priv-house-serv',
       'occupation_Prof-specialty', 'occupation_Protective-serv',
       'occupation_Sales', 'occupation_Tech-support',
       'occupation_Transport-moving', 'relationship_Not-in-family',
       'relationship_Other-relati

In [9]:

X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y,
    test_size=0.3,
    random_state=2137,
    stratify=y # gwarantuje, że proporcje klas w Twojej zmiennej docelowej (y) zostaną zachowane zarówno w zbiorze treningowym, jak i testowym
)
print(f"Zbiór treningowy: {X_train.shape[0]} obserwacji")
print(f"Zbiór testowy:    {X_test.shape[0]} obserwacji")

Zbiór treningowy: 21113 obserwacji
Zbiór testowy:    9049 obserwacji


AIC (Kryterium Informacyjne Akaike) i BIC (Bayesowskie Kryterium Informacyjne)

AIC i BIC wprowadzają karę za złożoność (liczbę zmiennych).

$\text{Kryterium} = -2 \cdot \log(\text{Wiarygodność}) + \text{Kara}$

W obu przypadkach: NIŻSZA WARTOŚĆ = LEPSZY MODEL.

Dla AIC $\text{Kara} = 2\text{liczba zmiennych}$

Dla BIC $\text{Kara} = \text{liczba zmiennych}\cdot\log{(\text{liczba obserwacji})}$

Metoda działa tak, że dopuki dodanie jakieś cechy polepsza model to dodajemy tą, która daje najelpszą wartość kryterium

In [10]:
def forward_selection(X, y, criterion='aic'):
    selected_features = []
    remaining_features = list(X.columns)

    # Model bazowy (tylko intercept / wyraz wolny)
    X_base_intercept = sm.add_constant(pd.DataFrame(index=X.index))
    base_model = sm.Logit(y, X_base_intercept).fit(disp=0) # disp żeby nic nie wyświetlało

    if criterion == 'aic':
        current_criterion_value = base_model.aic
    elif criterion == 'bic':
        current_criterion_value = base_model.bic

    print(f"Model bazowy | {criterion.upper()}: {current_criterion_value:.2f}")

    iteration = 1
    while True:
        best_value_in_iteration = current_criterion_value
        best_feature = None

        for feature in remaining_features:
            features_to_try = selected_features + [feature]
            X_temp = sm.add_constant(X[features_to_try].astype(float))

            try:
                model = sm.Logit(y, X_temp).fit_regularized(method='l1', # LASSO
                                                           alpha=0.2, # parametr lasso
                                                           disp=0,
                                                           maxiter=1000)

                if criterion == 'aic':
                    new_value = model.aic
                else:
                    new_value = model.bic

                if new_value < best_value_in_iteration:
                    best_value_in_iteration = new_value
                    best_feature = feature

            except (np.linalg.LinAlgError, ValueError):
                continue

        if best_feature is not None:
            selected_features.append(best_feature)
            remaining_features.remove(best_feature)
            current_criterion_value = best_value_in_iteration
            print(f"Iter {iteration}: Dodano '{best_feature}', Nowy {criterion.upper()}: {current_criterion_value:.2f}")
            iteration += 1
        else:
            print(f"\nIter {iteration}: Brak poprawy. KONIEC.")
            break

    return selected_features

In [11]:

import statsmodels.tools.sm_exceptions as sm_exceptions

# nie wiem dlaczego ale miałem baaardzo dużo warrningów, nie wiedziałem jak je naprawić, więc wszyski wyciszyłem
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    warnings.simplefilter("ignore", category=sm_exceptions.ConvergenceWarning)

    print("Selekcja AIC")
    selected_features_aic = forward_selection(X_train, y_train, criterion='aic')

    print("Selekcja BIC")
    selected_features_bic = forward_selection(X_train, y_train, criterion='bic')

print(f"Zestaw {len(selected_features_aic)} cech (AIC): {selected_features_aic}")
print(f"Ostateczny zestaw {len(selected_features_bic)} cech (BIC): {selected_features_bic}")

Selekcja AIC
Model bazowy | AIC: 23698.15
Iter 1: Dodano 'marital.status_Married-civ-spouse', Nowy AIC: 19252.75
Iter 2: Dodano 'education.num', Nowy AIC: 16785.83
Iter 3: Dodano 'capital.gain', Nowy AIC: 15390.71
Iter 4: Dodano 'age', Nowy AIC: 15124.73
Iter 5: Dodano 'hours.per.week', Nowy AIC: 14840.53
Iter 6: Dodano 'capital.loss', Nowy AIC: 14610.61
Iter 7: Dodano 'occupation_Farming-fishing', Nowy AIC: 14481.37
Iter 8: Dodano 'occupation_Exec-managerial', Nowy AIC: 14374.79
Iter 9: Dodano 'relationship_Own-child', Nowy AIC: 14308.42
Iter 10: Dodano 'occupation_Other-service', Nowy AIC: 14243.95
Iter 11: Dodano 'workclass_Self-emp-not-inc', Nowy AIC: 14199.30
Iter 12: Dodano 'occupation_Prof-specialty', Nowy AIC: 14161.26
Iter 13: Dodano 'occupation_Handlers-cleaners', Nowy AIC: 14128.23
Iter 14: Dodano 'relationship_Wife', Nowy AIC: 14106.24
Iter 15: Dodano 'sex_Male', Nowy AIC: 14015.39
Iter 16: Dodano 'occupation_Tech-support', Nowy AIC: 13996.32
Iter 17: Dodano 'marital.status

AIC

In [12]:
final_features = selected_features_aic

X_train_selected = X_train[final_features].astype(float)
X_test_selected = X_test[final_features].astype(float)

X_train_final = sm.add_constant(X_train_selected)
X_test_final = sm.add_constant(X_test_selected)

y_train = y_train.astype(int)

In [13]:
final_model = sm.Logit(y_train, X_train_final).fit_regularized(method='l1',
                                                               alpha=0.1,
                                                               disp=0,
                                                               maxiter=1000)
print(final_model.params)

/home/wiktor/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/home/wiktor/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


const                               -8.485100
marital.status_Married-civ-spouse    1.718576
education.num                        0.279961
capital.gain                         0.000319
age                                  0.026790
hours.per.week                       0.028995
capital.loss                         0.000642
occupation_Farming-fishing          -0.934274
occupation_Exec-managerial           0.815790
relationship_Own-child              -1.079549
occupation_Other-service            -0.677000
workclass_Self-emp-not-inc          -0.852983
occupation_Prof-specialty            0.665617
occupation_Handlers-cleaners        -0.707673
relationship_Wife                    1.241653
sex_Male                             0.832986
occupation_Tech-support              0.636152
marital.status_Never-married        -0.410619
occupation_Sales                     0.317613
occupation_Protective-serv           0.676497
occupation_Priv-house-serv          -4.580587
native.country_Columbia           

/home/wiktor/anaconda3/lib/python3.13/site-packages/statsmodels/base/l1_solvers_common.py:71: ConvergenceWarning: QC check did not pass for 21 out of 45 parameters
Try increasing solver accuracy or number of iterations, decreasing alpha, or switch solvers
  warnings.warn(message, ConvergenceWarning)
/home/wiktor/anaconda3/lib/python3.13/site-packages/statsmodels/base/l1_solvers_common.py:144: ConvergenceWarning: Could not trim params automatically due to failed QC check. Trimming using trim_mode == 'size' will still work.
  warnings.warn(msg, ConvergenceWarning)


In [14]:
print(final_model.summary())

                           Logit Regression Results                           
Dep. Variable:                 income   No. Observations:                21113
Model:                          Logit   Df Residuals:                    21068
Method:                           MLE   Df Model:                           44
Date:                Sat, 01 Nov 2025   Pseudo R-squ.:                  0.4194
Time:                        21:17:29   Log-Likelihood:                -6879.3
converged:                       True   LL-Null:                       -11848.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const                                -8.4851      0.255    -33.279      0.000      -8.985      -7.985
marital.status_Married-civ-spouse     1.7186      0.079     21

In [15]:
final_features_2 = selected_features_bic

X_train_selected_2 = X_train[final_features_2].astype(float)
X_test_selected_2 = X_test[final_features_2].astype(float)

X_train_final_2 = sm.add_constant(X_train_selected_2)
X_test_final_2 = sm.add_constant(X_test_selected_2)

y_train = y_train.astype(int)

final_model_2 = sm.Logit(y_train, X_train_final_2).fit_regularized(method='l1',
                                                               alpha=0.1,
                                                               disp=0,
                                                               maxiter=1000)
# print(final_model.params)

print(final_model_2.summary())

/home/wiktor/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/home/wiktor/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


                           Logit Regression Results                           
Dep. Variable:                 income   No. Observations:                21113
Model:                          Logit   Df Residuals:                    21090
Method:                           MLE   Df Model:                           22
Date:                Sat, 01 Nov 2025   Pseudo R-squ.:                  0.4147
Time:                        21:25:04   Log-Likelihood:                -6934.9
converged:                       True   LL-Null:                       -11848.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const                                -8.6569      0.197    -43.952      0.000      -9.043      -8.271
marital.status_Married-civ-spouse     1.7285      0.079     21

/home/wiktor/anaconda3/lib/python3.13/site-packages/statsmodels/base/l1_solvers_common.py:71: ConvergenceWarning: QC check did not pass for 9 out of 23 parameters
Try increasing solver accuracy or number of iterations, decreasing alpha, or switch solvers
  warnings.warn(message, ConvergenceWarning)
/home/wiktor/anaconda3/lib/python3.13/site-packages/statsmodels/base/l1_solvers_common.py:144: ConvergenceWarning: Could not trim params automatically due to failed QC check. Trimming using trim_mode == 'size' will still work.
  warnings.warn(msg, ConvergenceWarning)
